# 04. Model Diagnostics & Operational Alert Tuning

Granular evaluation of candidate classifiers on chronological event-centred folds. Each model is evaluated with Accuracy, TPR (Recall), FPR, Balanced Accuracy, ConfusionMatrixDisplay, side-by-side ROC and PR curves, MCC, and master comparison tables.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    auc,
    roc_curve,
    roc_auc_score,
    matthews_corrcoef,
    balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from src.data.load_data import load_workbook
from src.data.split_data import event_centric_folds
from src.utils.io import load_config

sns.set_theme(style="whitegrid")
config = load_config()
df_features = pd.read_csv(config["project"]["featured_file"], parse_dates=["Timestamp"])
workbook = load_workbook(config["project"]["raw_file"])
events = workbook["events"]


## 1. Chronological Event-Centred Splitting Setup


In [ ]:
folds = event_centric_folds(df_features, events, pre_event_hours=72, post_event_hours=24)
valid_folds = [f for f in folds if f.train["target_failure"].nunique() >= 2]
test_fold = valid_folds[-1]

excluded = {"target_failure", "Machine ID", "Timestamp", "Event Timestamp"}
feature_cols = [c for c in test_fold.train.columns if c not in excluded and pd.api.types.is_numeric_dtype(test_fold.train[c])]

X_train = test_fold.train[feature_cols]
y_train = test_fold.train["target_failure"].astype(int)
X_test = test_fold.test[feature_cols]
y_test = test_fold.test["target_failure"].astype(int)

print(f"Evaluation Fold: {test_fold.description}")
print(f"Train Shape: {X_train.shape}, Class Balance: {dict(y_train.value_counts())}")
print(f"Test Shape: {X_test.shape}, Class Balance: {dict(y_test.value_counts())}")


## 2. Granular Model Training & Diagnostic Benchmarks

We benchmark candidate models with operational decision threshold tuning ($P \ge 0.10$) to account for rare failure events.


In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=500, random_state=42))
    ]),
    "DecisionTree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=42))
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample", random_state=42))
    ]),
    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=42))
    ]),
    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(n_estimators=150, class_weight="balanced", random_state=42))
    ]),
    "AdaBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(random_state=42))
    ])
}

THRESHOLD = 0.10
results_list = []
model_predictions = {}
model_probabilities = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    
    model_predictions[name] = pred
    model_probabilities[name] = proba
    
    acc = accuracy_score(y_test, pred)
    bal_acc = balanced_accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    fpr_arr, tpr_arr, _ = roc_curve(y_test, proba)
    roc_auc_val = auc(fpr_arr, tpr_arr)
    
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, proba)
    pr_auc_val = auc(rec_arr, prec_arr)
    mcc_val = matthews_corrcoef(y_test, pred)
    
    results_list.append({
        "Model": name,
        "Accuracy": acc,
        "Balanced_Acc": bal_acc,
        "ROC-AUC": roc_auc_val,
        "PR-AUC": pr_auc_val,
        "MCC": mcc_val,
        "TPR (Recall)": tpr,
        "FPR": fpr,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })
    
    print(f"\n=======================================================")
    print(f"MODEL: {name.upper()} (Threshold = {THRESHOLD})")
    print(f"=======================================================")
    print(f"Accuracy: {acc:.4f} | Balanced Acc: {bal_acc:.4f} | ROC-AUC: {roc_auc_val:.4f} | PR-AUC: {pr_auc_val:.4f} | MCC: {mcc_val:.4f}")
    print(f"TPR (Recall): {tpr:.4f} | FPR: {fpr:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, pred, zero_division=0))
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 4.5))
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(ax=ax1, cmap="viridis", colorbar=False)
    ax1.set_title(f"{name} - Confusion Matrix")
    
    ax2.plot(fpr_arr, tpr_arr, label=f"ROC-AUC = {roc_auc_val:.4f}", color="darkblue")
    ax2.plot([0, 1], [0, 1], "k--", label="Random")
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title(f"{name} - ROC Curve")
    ax2.legend()
    ax2.grid(True)
    
    ax3.plot(rec_arr, prec_arr, label=f"PR-AUC = {pr_auc_val:.4f}", color="darkred")
    ax3.set_xlabel("Recall")
    ax3.set_ylabel("Precision")
    ax3.set_title(f"{name} - Precision-Recall Curve")
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout()
    plt.show()


## 3. Master Model Performance Comparison


In [ ]:
summary_df = pd.DataFrame(results_list)
print("=== MASTER BASELINE CLASSIFIER COMPARISON (THRESHOLD = 0.10) ===")
display(summary_df)

best_prauc = summary_df.loc[summary_df["PR-AUC"].idxmax()]
print(f"\nBEST PERFORMING MODEL (PR-AUC): {best_prauc['Model']} (PR-AUC: {best_prauc['PR-AUC']:.4f}, ROC-AUC: {best_prauc['ROC-AUC']:.4f})")


## 4. Multi-Model Overlay Curves (ROC & PR Curves Side-by-Side)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot([0, 1], [0, 1], "k--", label="Random Baseline")
for name, proba in model_probabilities.items():
    fpr_arr, tpr_arr, _ = roc_curve(y_test, proba)
    score = auc(fpr_arr, tpr_arr)
    ax1.plot(fpr_arr, tpr_arr, label=f"{name} ({score:.3f})")

ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curves - All Candidate Classifiers", fontsize=13)
ax1.legend(loc="lower right")
ax1.grid(True)

for name, proba in model_probabilities.items():
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, proba)
    score = auc(rec_arr, prec_arr)
    ax2.plot(rec_arr, prec_arr, label=f"{name} ({score:.3f})")

ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curves - All Candidate Classifiers", fontsize=13)
ax2.legend(loc="upper right")
ax2.grid(True)

plt.tight_layout()
plt.show()


## 5. Operational Threshold Tuning & Warning Lead Time Analysis


In [ ]:
hist_model = models["HistGradientBoosting"]
proba = model_probabilities["HistGradientBoosting"]

threshold_sweep = []
for t in np.linspace(0.01, 0.50, 50):
    p = (proba >= t).astype(int)
    cm = confusion_matrix(y_test, p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    threshold_sweep.append({"Threshold": t, "Recall (TPR)": tpr, "FPR": fpr})

sweep_df = pd.DataFrame(threshold_sweep)

plt.figure(figsize=(10, 5))
plt.plot(sweep_df["Threshold"], sweep_df["Recall (TPR)"], label="Recall (TPR / Coverage)", color="green", linewidth=2)
plt.plot(sweep_df["Threshold"], sweep_df["FPR"], label="False Positive Rate (Alarm Fatigue)", color="crimson", linewidth=2)
plt.axvline(0.10, color="black", linestyle="--", label="Operational Cutoff (0.10)")
plt.title("Alert Decision Threshold Optimization (Coverage vs False Alarm Cost)", fontsize=13)
plt.xlabel("Probability Decision Threshold")
plt.ylabel("Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
